In [35]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [36]:
import os
import sys
import numpy as np
from torchvision.transforms.v2 import RandAugment
import torch
from tqdm.notebook import tqdm
from torch.utils.tensorboard import SummaryWriter
import torch.nn as nn

sys.path.append(os.path.abspath("../src"))
sys.path.append(os.path.abspath("../models"))

from dataset import get_data_loaders
from backbone import BackBone
from multiheadmodel import MultiHeadModel
from utils import deterministic, train

In [37]:
backbone= BackBone()
backbone.load_state_dict(torch.load("../models/weights/backbone.pth"))
model = MultiHeadModel(backbone)
model.add_head(0, 2)

/Users/marcelokaucher/miniforge3/envs/vision/lib/python3.14/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/marcelokaucher/miniforge3/envs/vision/lib/python3.14/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Comentario de claude para acelerar el entrenamiento de las cabezas:

Si el backbone está congelado, podés pre-calcular los embeddings una sola vez y entrenar solo sobre ellos — mucho más rápido

In [38]:
for param in model.backbone.parameters(): 
    param.requires_grad = False

In [39]:
dataloaders = get_data_loaders(batch_size=512)
task0_train = dataloaders[0][0]
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

/Users/marcelokaucher/miniforge3/envs/vision/lib/python3.14/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Task [0, 1]: Train=9000, Val=1000, Test=2000
Task [2, 3]: Train=9000, Val=1000, Test=2000
Task [4, 5]: Train=9000, Val=1000, Test=2000
Task [6, 7]: Train=9000, Val=1000, Test=2000
Task [8, 9]: Train=9000, Val=1000, Test=2000


MultiHeadModel(
  (backbone): BackBone(
    (encoder): ResNet(
      (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (maxpool): Identity()
      (layer1): Sequential(
        (0): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu): ReLU(inplace=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
        (1): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_runn

In [41]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

epochs = 5

# modularizar
# for epoch in tqdm(range(epochs), desc = "Epochs", unit = "epoch"):
#     batch_bar = tqdm(task0_train, desc = f"Epoch {epoch+1}/{epochs}", leave=False, unit="batch")
#     for i, (x, y) in enumerate(batch_bar):
#         x_all, y_all = x.to(device), y.to(device) # ver randaugment

#         optimizer.zero_grad()
#         pred = model(x_all, 0)
#         loss = criterion(pred, y_all)
#         writer.add_scalar("Loss/Train", loss.item(), epoch * len(task0_train) + i)
#         print(loss.item())

#         loss.backward()
#         optimizer.step()

#         batch_bar.set_postfix({"loss": loss.item()})
#         break
    
# writer.close()
# model.save("../models/weights/task0_training.pth")
train(model, dataloaders[0][0], optimizer, criterion, "task0_4_2", epochs, 0)

Epochs:   0%|          | 0/5 [00:00<?, ?epoch/s]

Epoch 1/5:   0%|          | 0/18 [00:00<?, ?batch/s]

0.729310154914856
0.6925929188728333
0.711601734161377
0.6574608683586121
0.6602023839950562
0.6956092119216919
0.7100773453712463
0.7081817388534546
0.6824069619178772
0.6986609697341919
0.7049841284751892
0.7079557776451111
0.6877471208572388
0.6687970161437988
0.6734303832054138
0.6827881932258606
0.6675941348075867
0.6902680993080139


Epoch 2/5:   0%|          | 0/18 [00:00<?, ?batch/s]

0.6736447215080261
0.6745721697807312
0.6786595582962036
0.6784390211105347
0.6566445827484131
0.6820148825645447
0.6784656047821045
0.6967030763626099
0.6669210195541382
0.6600773930549622
0.6513926982879639
0.6742235422134399
0.6975829005241394
0.6819034218788147
0.6660140752792358
0.6870896220207214
0.6643415093421936


KeyboardInterrupt: 

In [ ]:
task0_val = dataloaders[0][1]

model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for x, y in tqdm(task0_val, desc="Evaluating", unit="batch"):
        x, y = x.to(device), y.to(device)
        pred = model(x, 0)
        _, predicted = torch.max(pred.data, 1)
        total += y.size(0)
        correct += (predicted == y).sum().item()
    print(f"Accuracy: {100 * correct / total:.2f}%")

Evaluating:   0%|          | 0/2 [00:00<?, ?batch/s]

Accuracy: 49.20%
